# Semantic-typing self-gate — live demo

The harness proves code three ways before pasting it: **it runs** (`_run_in_kernel`), **the value is right** (`_assert_report`), and now **the types check** (`_typecheck`, running real mypy). This notebook demos the third oracle on a **wrong-typed function** — one that runs fine and has a `->`, but whose annotation is a lie.

In [1]:
import contextlib, io, os, re, sys
os.environ.setdefault('HARNESS_MODEL_SERVER', 'http://127.0.0.1:7777')
try:
    from harness import lab
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
    from harness import lab

def chat_quietly(message, code):
    """Run the real _chat but swallow the live thinking-stream markers so the demo reads clean."""
    with contextlib.redirect_stdout(io.StringIO()):
        return lab._chat([{'role': 'user', 'content': message}], context=code, intent='code', think='brief')

def show_reply(raw):
    return re.sub(r'@@HARNESS_\w+@@\S*|__H\w+__\S*', '', raw).strip()

def code_of(raw):
    blocks = re.findall(r'```(?:python)?\s*\n(.*?)```', show_reply(raw), re.S)
    return blocks[-1].strip() if blocks else ''

## 1. The oracle catches a wrong-typed function
`add` returns `a + b` (an `int`) but is annotated `-> str`. It **runs** and it **has a `->`** — only a type checker exposes the lie.

In [2]:
wrong = 'def add(a: int, b: int) -> str:\n    return a + b'
print(wrong, '\n')

r = lab._typecheck(wrong)
print('type-checks? ', r['ok'])
for e in r['errors']:
    print('  mypy:', e)

def add(a: int, b: int) -> str:
    return a + b 



type-checks?  False
  mypy: Incompatible return value type (got "int", expected "str")  [return-value]


## 2. Fix it through the REAL model (E2B)
"fix the type hints on this" routes through the same `_chat` the notebook chat uses. The gate runs mypy on the model's output before it is pasted, so whatever lands here is **guaranteed to type-check**.

In [3]:
raw = chat_quietly('fix the type hints on this', wrong)
fixed = code_of(raw)
print(fixed, '\n')
print('gate verdict:', lab._typecheck_summary(lab._typecheck(fixed)))

def add(a: int, b: int) -> int:
    return a + b 



gate verdict: types check out


## 3. The retry + honest-flag (simulated stubborn model)
The real E2B is too good to mis-type simple code, so to show the gate's **corrective loop** we temporarily make the model *stubbornly* return the wrong type. Only `_generate` is stubbed — **mypy and the gate logic are real**. The gate retries twice, can't get a clean check, and then **inserts the best attempt but says so honestly** rather than pretending the types are right.

In [4]:
orig_generate, orig_typecheck = lab._generate, lab._typecheck
mypy_calls = []
def counting_typecheck(code, *a, **k):
    res = orig_typecheck(code, *a, **k)
    mypy_calls.append('ok' if res['ok'] else 'FAIL')
    return res
def stubborn_generate(messages, *a, **k):
    return '```python\ndef add(a: int, b: int) -> str:\n    return a + b\n```'

lab._generate, lab._typecheck = stubborn_generate, counting_typecheck
try:
    with contextlib.redirect_stdout(io.StringIO()):
        raw = lab._chat([{'role': 'user', 'content': 'add type hints'}], context=wrong, intent='code', think='brief')
finally:
    lab._generate, lab._typecheck = orig_generate, orig_typecheck

print('gate ran mypy', len(mypy_calls), 'times:', mypy_calls, '\u2192 never clean')
print('mode:', 'ASK' if raw.startswith(lab._MODE_ASK) else 'CODE (inserted, with a warning)')
print()
print(show_reply(raw))

gate ran mypy 3 times: ['FAIL', 'FAIL', 'FAIL'] → never clean
mode: CODE (inserted, with a warning)

I added the type hints, but they don't type-check — mypy says: `Incompatible return value type (got "int", expected "str")  [return-value]`. I've inserted my best attempt; want me to adjust the annotations (or the code) so the types line up?

```python
def add(a: int, b: int) -> str:
    return a + b

```


**What you just saw:** the gate type-checked the model's output, retried twice feeding mypy's own error back, and — since the (stubborn) model never fixed it — pasted the best attempt with an honest *"these don't type-check"* note instead of pretending. With the real model (cell 2) the first attempt already checks, so it just pastes clean. Either way: **the harness never silently ships a type it hasn't verified.**